In [ ]:
#| default_exp cli

# cli

> `panjika log`, `panjika trail`, `panjika landed`.

The command line is the second half of the agent-facing API. Every query takes `--json`, so a
harness with no Python in it can still ask what happened and read the answer, and a model can
be handed the output of `panjika landed --json` without a wrapper.

In [ ]:
#| export
import argparse, json, os, sys, time
from pathlib import Path

from fastcore.basics import AttrDict
from fastcore.foundation import L

from panjika.core import DETAIL, LEDGER, Home, dumps, records
from panjika.git import blend, landed, link_commit, report
from panjika.harness import ADAPTERS, hook, ingest, install
from panjika.read import Ledger, _since

## Printing

One rule: `--json` prints the records, anything else prints a line per row. Nothing is coloured
unless a terminal is on the other end, so a pipe gets clean text.

`plain` names `L` in its `isinstance` check. `L` is not a `list` subclass, so dropping it there would send every `L` in a record through the fallback and break `--json`.

In [ ]:
#| export
STATE_COLOURS = {'landed': '32', 'partly_landed': '33', 'pending': '36', 'replaced': '31',
                 'gone': '31', 'untracked': '90', 'uncertain': '33', 'unknown': '90'}


def tint(text, code, on=None):
    "Wrap `text` in an ANSI colour when stdout is a terminal."
    if on is None: on = sys.stdout.isatty()
    return f'\033[{code}m{text}\033[0m' if on and code else str(text)


def ago(at):
    "How long ago, in the shortest form that is still true."
    secs = max(0, time.time() - (at or 0))
    for n, unit in ((86400 * 365, 'y'), (86400 * 30, 'mo'), (86400, 'd'), (3600, 'h'), (60, 'm')):
        if secs >= n: return f'{int(secs // n)}{unit} ago'
    return 'just now'


def plain(o):
    "Anything the ledger holds, as something JSON can carry."
    if isinstance(o, Path): return str(o)
    if isinstance(o, dict): return {str(k): plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set, L)): return [plain(x) for x in o]
    if hasattr(o, 'dict') and callable(o.dict): return plain(o.dict())
    return o


def out_json(obj):
    "Print one JSON document, for a caller that is not a person."
    sys.stdout.write(dumps(plain(obj)).decode())

## The commands

In [ ]:
#| export
def cmd_log(a):
    "Sessions, newest first. The log."
    led = Ledger(a.home, a.root)
    rows = led.sessions(limit=a.limit, harness=a.harness, since=a.since, path=a.path, repo=a.repo)
    if a.json: return out_json([dict(led.session(r.session)) for r in rows])
    if not rows: return print('no sessions recorded')
    for r in rows:
        row = led.session(r.session)
        head = tint(r.session[:12], '33')
        who = tint(f"{r.get('harness', '?')}/{r.get('model') or '?'}", '36')
        files = ', '.join(t.path for t in row.files[:3]) or 'no files'
        if len(row.files) > 3: files += f' +{len(row.files) - 3}'
        fail = tint(f"  {row.steps_fail} failed", '31') if row.steps_fail else ''
        print(f"{head}  {who}  {ago(r.get('started') or r.get('at'))}")
        print(f"    {(r.get('title') or r.get('prompt') or '(no prompt recorded)')[:100]}")
        print(f"    {row.n_steps} steps{fail}  ·  {files}")


def cmd_show(a):
    "One session, whole."
    led = Ledger(a.home, a.root)
    sid = a.session
    if sid in ('', 'latest', None):
        rows = led.sessions(limit=1)
        if not rows: return print('no sessions recorded')
        sid = rows[0].session
    row = led.session(sid)
    if a.json: return out_json(dict(row))
    print(tint(f"{row.session}  {row.get('harness', '?')}/{row.get('model') or '?'}", '33'))
    print(f"  {row.get('title') or row.get('prompt') or ''}")
    print(f"  {row.get('repo', '')} {row.get('branch', '')}  {ago(row.get('started') or row.get('at'))}  "
          f"{row.n_steps} steps, {row.steps_fail} failed")
    for p, origin in row.get('prompts', [])[1:]:
        mark = tint('>', '36') if origin != 'injected' else tint('~', '90')
        print(f"  {mark} {' '.join(str(p).split())[:88]}")
    for t in row.files: print(f"  {tint('~', '35')} {t.path}  +{t.get('added', 0)}/-{t.get('removed', 0)}")
    for c in row.commits: print(f"  {tint(c.get('sha', '')[:7], '32')} {c.get('subject', '')}")
    for s in row.steps[-a.limit:]:
        mark = ' ' if s.get('ok', True) else tint('!', '31')
        print(f"  {mark} {s.get('tool', ''):<16} {s.get('target', '')[:70]}")

In [ ]:
#| export
def cmd_trail(a):
    "Every session and every commit that ever touched one file."
    rows = blend(a.path, home=a.home, start=a.root, limit=a.limit)
    if a.json: return out_json([dict(r) for r in rows])
    if not rows: return print(f'nothing recorded for {a.path}')
    for r in rows:
        if r.kind == 'commit':
            print(f"{tint(r.short, '32')}  {tint('commit', '90')}  {r.title[:80]}")
            print(f"          {r.author}  {ago(r.at)}")
        else:
            print(f"{tint(r.session[:7], '33')}  {tint(r.harness or 'agent', '36')}  {r.title[:80]}")
            where = f"  {r.get('branch')}" if r.get('branch') else ''
            print(f"          +{r.added}/-{r.removed}  {r.model or ''}{where}  {ago(r.at)}")


def cmd_landed(a):
    "What became of what a session wrote."
    r = report(a.session, path=a.path, home=a.home, start=a.root)
    if a.json: return out_json({'summary': r.summary, 'counts': r.counts,
                                'verdicts': [v.dict() for v in r.verdicts]})
    if not r.verdicts: return print('nothing recorded to check')
    for v in r.verdicts:
        share = '' if not v.total else f'  {v.kept}/{v.total} lines'
        where = tint(f"  on {v.branch}", '90') if v.branch else ''
        print(f"{tint(v.state, STATE_COLOURS.get(v.state, ''))}  {v.path}{share}{where}")
        print(f"    {v.why}")
        for c in v.commits: print(f"    {tint(c.short, '32')} {c.subject[:70]}")
    print(f'\n{r.summary}')

Standing in a repository means that repository's ledger, whatever directory the session ran in. `--all` is the one case that routes each session to its own project instead.

`export` writes bytes wherever stdout has a byte stream, which is every real invocation, and text where a caller has replaced stdout with a stream that has none. `dumps` already ends every record with its newline.

In [ ]:
#| export
def cmd_init(a):
    home = Home(a.home, a.root).init()
    print(f'ledger at {home.path}')


def cmd_install(a):
    res = install(a.root, claude_code=not a.no_claude_code, codex=not a.no_codex,
                  git=not a.no_git, skill=not a.no_skill, home=a.home)
    for p in res.wrote: print(f'wrote {p}')
    for s in res.skipped: print(f'skipped {s}')
    print(f'ledger at {res.home}')
    if not a.no_codex: print(f'\n{res.note}')


def cmd_hook(a):
    hook(a.adapter, home=a.home)


def cmd_record(a):
    payload = json.loads(a.json_text) if a.json_text else json.loads(sys.stdin.read() or '{}')
    p = ingest(payload, a.adapter, a.home, a.root)
    print(f'{p.wrote} record(s) for {p.session}')


def cmd_link(a):
    linked = link_commit(a.sha, home=a.home, start=a.root)
    print(f"{a.sha[:7]} -> {', '.join(linked) if linked else 'no session touched these files'}")


def cmd_stats(a):
    s = Ledger(a.home, a.root).stats()
    if a.json: return out_json(dict(s))
    for k, v in s.items(): print(f'{k:<12} {v}')


def cmd_backfill(a):
    "Read transcripts a harness wrote before there was a ledger."
    from .backfill import backfill, backfill_session
    home = None if a.all else Home(a.home, a.root)
    p = Path(a.path).expanduser() if a.path else None
    if p is not None and p.is_file(): rows = backfill_session(p, home=home, start=a.root)
    elif p is not None: rows = backfill(root=p, home=home, start=a.root)
    else: rows = backfill(cwd=None if a.all else a.root, home=home, start=a.root)
    if a.json: return out_json([{'session': s, 'records': n} for s, n in rows])
    for s, n in rows: print(f'{n:6}  {s}')
    print(f'{len(rows)} session(s), {sum(n for _, n in rows)} records')


def cmd_export(a):
    "Every ledger record as JSONL on stdout. The portable form."
    led = Ledger(a.home, a.root)
    after = _since(a.since)
    raw = getattr(sys.stdout, 'buffer', None)
    for r in led.all(DETAIL if a.detail else LEDGER):
        if (r.get('at') or 0) < after: continue
        line = dumps(dict(r))
        raw.write(line) if raw else sys.stdout.write(line.decode())

## Wiring

`panjika trail x | head` closes the pipe under us. Writing to `devnull` from there on keeps the interpreter from reporting the same broken pipe again while it exits.

In [ ]:
#| export
def build_parser():
    "The whole command line."
    p = argparse.ArgumentParser(prog='panjika', description='the register of agent deeds')
    p.add_argument('--home', default=os.environ.get('PANJIKA_HOME'),
                   help='the ledger folder. Default finds one from --root')
    p.add_argument('--root', default='.', help='where to look for a ledger and a repository')
    sub = p.add_subparsers(dest='cmd', required=True)

    def add(name, fn, help, json_flag=True):
        s = sub.add_parser(name, help=help)
        s.set_defaults(fn=fn)
        if json_flag: s.add_argument('--json', action='store_true', help='print records, not prose')
        return s

    s = add('log', cmd_log, 'sessions, newest first')
    s.add_argument('-n', '--limit', type=int, default=20)
    s.add_argument('--harness', default=''); s.add_argument('--since', default='')
    s.add_argument('--path', default=''); s.add_argument('--repo', default='')

    s = add('show', cmd_show, 'one session, whole')
    s.add_argument('session', nargs='?', default='latest')
    s.add_argument('-n', '--limit', type=int, default=40)

    s = add('trail', cmd_trail, 'every session and commit that touched one file')
    s.add_argument('path'); s.add_argument('-n', '--limit', type=int, default=40)

    s = add('landed', cmd_landed, 'what became of what a session wrote')
    s.add_argument('session', nargs='?', default='latest')
    s.add_argument('--path', default='')

    add('init', cmd_init, 'make a ledger here', json_flag=False)
    s = add('install', cmd_install, 'write the harness hooks', json_flag=False)
    s.add_argument('--no-claude-code', action='store_true')
    s.add_argument('--no-codex', action='store_true')
    s.add_argument('--no-git', action='store_true')
    s.add_argument('--no-skill', action='store_true')

    s = add('hook', cmd_hook, 'record one harness payload from stdin', json_flag=False)
    s.add_argument('adapter', choices=sorted(ADAPTERS), nargs='?', default='generic')

    s = add('record', cmd_record, 'record one payload given on the command line', json_flag=False)
    s.add_argument('json_text', nargs='?', default='')
    s.add_argument('--adapter', choices=sorted(ADAPTERS), default='generic')

    s = add('link-commit', cmd_link, 'link a commit to the sessions that earned it', json_flag=False)
    s.add_argument('sha', nargs='?', default='HEAD')

    s = add('backfill', cmd_backfill, 'read transcripts written before this ledger existed')
    s.add_argument('path', nargs='?', default='',
                   help='a transcript, or a folder of them. Default is this project')
    s.add_argument('--all', action='store_true', help='every project, not just this one')

    add('stats', cmd_stats, 'what this ledger holds')
    s = add('export', cmd_export, 'every record as JSONL on stdout', json_flag=False)
    s.add_argument('--since', default=''); s.add_argument('--detail', action='store_true')
    return p


def main(argv=None):
    a = build_parser().parse_args(argv)
    try: a.fn(a)
    except BrokenPipeError:
        sys.stdout = open(os.devnull, 'w')
        return 0
    return 0

## Trying it

In [ ]:
import subprocess, tempfile
from fastcore.test import test_eq
from panjika.write import Scribe

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'T')
(d/'app.py').write_text('X = 1\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'first')

sc = Scribe(home=d/'.panjika', start=d); sc.home.init()
sc.begin('claude-code', model='opus-5', prompt='bump the constant')
(d/'app.py').write_text('X = 2\n')
sc.touch(d/'app.py', 'edit', sc.step('Edit', target='app.py'))
sc.end('done')

main(['--home', str(d/'.panjika'), '--root', str(d), 'log'])

In [ ]:
main(['--home', str(d/'.panjika'), '--root', str(d), 'landed'])

In [ ]:
_git(d, 'commit', '-aqm', 'bump it')
main(['--home', str(d/'.panjika'), '--root', str(d), 'link-commit'])
main(['--home', str(d/'.panjika'), '--root', str(d), 'trail', 'app.py'])

In [ ]:
main(['--home', str(d/'.panjika'), '--root', str(d), 'landed'])

`--json` is what a model is handed, so it has to parse and it has to carry the reason and the commits rather than the bare state. Nothing is coloured when the far end is not a terminal, and every state the verdict can take has a colour, because one the renderer does not know would print undecorated and read as an afterthought.

In [ ]:
import io, json
from contextlib import redirect_stdout
from panjika.git import STATES

def _run(*args):
    buf = io.StringIO()
    with redirect_stdout(buf): main(['--home', str(d/'.panjika'), '--root', str(d), *args])
    return buf.getvalue()

doc = json.loads(_run('landed', '--json'))
test_eq(doc['counts'], {'landed': 1})
v = doc['verdicts'][0]
test_eq((v['state'], v['path'], v['evidence'], v['survived']), ('landed', 'app.py', 'lines', 1.0))
test_eq(v['commits'][0]['subject'], 'bump it')
assert v['why']

test_eq(json.loads(_run('log', '--json'))[0]['files'][0]['path'], 'app.py')
test_eq(sorted({r['kind'] for r in json.loads(_run('trail', 'app.py', '--json'))}),
        ['commit', 'session'])
out = Path(tempfile.mkdtemp())/'ledger.jsonl'
with out.open('w') as f, redirect_stdout(f): main(
    ['--home', str(d/'.panjika'), '--root', str(d), 'export'])
kinds = sorted({json.loads(l)['kind'] for l in out.read_text().splitlines() if l.strip()})
test_eq(kinds, ['commit', 'session', 'step', 'touch'])
test_eq(sorted({json.loads(l)['kind'] for l in _run('export').splitlines() if l.strip()}), kinds)

for args in (['show'], ['show', sc.session], ['stats'], ['trail', 'app.py']):
    test_eq(main(['--home', str(d/'.panjika'), '--root', str(d), *args]), 0)
_run('record', json.dumps({'session': 'x-1', 'do': 'begin', 'harness': 'a script'}))
test_eq(json.loads(_run('log', '--json'))[0]['harness'], 'a script')

assert '\033[' not in _run('landed')
assert set(STATES) <= set(STATE_COLOURS)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()